<a href="https://www.kaggle.com/code/shedai/optiprophet-benchmarks?scriptVersionId=270086844" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Gerekli tüm kütüphanelerin kurulumu
# 'optitime-prophet', proje dosyalarınızda belirtilen paket adıdır.
!pip install -q optitime-prophet
!pip install -q prophet
!pip install -q neuralprophet
!pip install -q pmdarima

print("Kütüphaneler kuruldu.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [3]:
import pandas as pd
import numpy as np
import logging
from optitime import OptiProphet, load_dataset
from prophet import Prophet
from neuralprophet import NeuralProphet
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# Kütüphanelerin ayrıntılı log çıktılarını bastır
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('neuralprophet').setLevel(logging.WARNING)

print("Kütüphaneler içe aktarıldı.")

Kütüphaneler içe aktarıldı.


In [4]:
def get_metrics(y_true, y_pred):
    """
    Gerçek ve tahmin edilen değerler arasındaki metrikleri hesaplar.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

def train_test_split_ts(df, test_periods):
    """
    Zaman serisi verisini eğitim ve test setlerine ayırır.
    """
    train = df.iloc[:-test_periods].copy()
    test = df.iloc[-test_periods:].copy()
    return train, test, test['y']

In [5]:
# Sonuçları saklamak için bir liste
results = []
TEST_PERIODS_AIR = 24  # Air Passengers için 24 ay (2 yıl)

# *** HATA DÜZELTMESİ BURADA ***
# OptiProphet'in min_history=30 gereksinimi için test periyodunu 12'den 6'ya düşürdük.
# Toplam veri 36 -> Eğitim = 30, Test = 6.
TEST_PERIODS_SHAMPOO = 6 

# ---------------------------------------------------------------------------
# Veri Seti 1: Air Passengers
# ---------------------------------------------------------------------------
print("Veri Seti 1: Air Passengers çalıştırılıyor...")
dataset_name = "air_passengers"
df_air = load_dataset(dataset_name)

# Bu veri seti çarpımsaldır. Log dönüşümü tüm modeller için daha adil bir karşılaştırma sağlar.
df_air['y_orig'] = df_air['y']
df_air['y'] = np.log(df_air['y'])

train_df, test_df, y_true_log = train_test_split_ts(df_air, TEST_PERIODS_AIR)
y_true_orig = test_df['y_orig'] # Metrikler için orijinal ölçek

# --- Model 1: OptiProphet ---
m_opti = OptiProphet(ar_order=12, ma_order=1, n_changepoints=25)
m_opti.fit(train_df[['ds', 'y']])
future_opti = m_opti.make_future_dataframe(periods=TEST_PERIODS_AIR)
forecast_opti = m_opti.predict(future_opti)
y_pred_log = forecast_opti.iloc[-TEST_PERIODS_AIR:]['yhat']
# Metrikler için log dönüşümünü geri al
metrics_opti = get_metrics(y_true_orig, np.exp(y_pred_log))
results.append({"Dataset": dataset_name, "Model": "OptiProphet", **metrics_opti})

# --- Model 2: Prophet (Default) ---
m_prophet = Prophet()
m_prophet.fit(train_df[['ds', 'y']])
future_prophet = m_prophet.make_future_dataframe(periods=TEST_PERIODS_AIR)
forecast_prophet = m_prophet.predict(future_prophet)
y_pred_log = forecast_prophet.iloc[-TEST_PERIODS_AIR:]['yhat']
metrics_prophet = get_metrics(y_true_orig, np.exp(y_pred_log))
results.append({"Dataset": dataset_name, "Model": "Prophet", **metrics_prophet})

# --- Model 3: NeuralProphet ---
m_neural = NeuralProphet()
m_neural.fit(train_df[['ds', 'y']], freq="MS") # Aylık frekans
future_neural = m_neural.make_future_dataframe(train_df[['ds', 'y']], periods=TEST_PERIODS_AIR)
forecast_neural = m_neural.predict(future_neural)
y_pred_log = forecast_neural.iloc[-TEST_PERIODS_AIR:]['yhat1']
metrics_neural = get_metrics(y_true_orig, np.exp(y_pred_log))
results.append({"Dataset": dataset_name, "Model": "NeuralProphet", **metrics_neural})

# --- Model 4: Auto-ARIMA ---
# SARIMA (Mevsimsel ARIMA) için m=12
m_arima = auto_arima(train_df['y'], seasonal=True, m=12, suppress_warnings=True, stepwise=True)
y_pred_log = m_arima.predict(n_periods=TEST_PERIODS_AIR)
metrics_arima = get_metrics(y_true_orig, np.exp(y_pred_log))
results.append({"Dataset": dataset_name, "Model": "Auto-ARIMA", **metrics_arima})

print("Air Passengers tamamlandı.")

# ---------------------------------------------------------------------------
# Veri Seti 2: Shampoo Sales
# ---------------------------------------------------------------------------
print("\nVeri Seti 2: Shampoo Sales çalıştırılıyor...")
dataset_name = "shampoo_sales"
df_shampoo = load_dataset(dataset_name)
# Bu veri seti log dönüşümüne ihtiyaç duymaz.

train_df, test_df, y_true = train_test_split_ts(df_shampoo, TEST_PERIODS_SHAMPOO)

# --- Model 1: OptiProphet ---
m_opti = OptiProphet(ar_order=3, ma_order=1) # Basit bir ARMA
m_opti.fit(train_df[['ds', 'y']])
future_opti = m_opti.make_future_dataframe(periods=TEST_PERIODS_SHAMPOO)
forecast_opti = m_opti.predict(future_opti)
y_pred = forecast_opti.iloc[-TEST_PERIODS_SHAMPOO:]['yhat']
metrics_opti = get_metrics(y_true, y_pred)
results.append({"Dataset": dataset_name, "Model": "OptiProphet", **metrics_opti})

# --- Model 2: Prophet (Default) ---
m_prophet = Prophet()
m_prophet.fit(train_df[['ds', 'y']])
future_prophet = m_prophet.make_future_dataframe(periods=TEST_PERIODS_SHAMPOO)
forecast_prophet = m_prophet.predict(future_prophet)
y_pred = forecast_prophet.iloc[-TEST_PERIODS_SHAMPOO:]['yhat']
metrics_prophet = get_metrics(y_true, y_pred)
results.append({"Dataset": dataset_name, "Model": "Prophet", **metrics_prophet})

# --- Model 3: NeuralProphet ---
m_neural = NeuralProphet()
m_neural.fit(train_df[['ds', 'y']], freq="MS")
future_neural = m_neural.make_future_dataframe(train_df[['ds', 'y']], periods=TEST_PERIODS_SHAMPOO)
forecast_neural = m_neural.predict(future_neural)
y_pred = forecast_neural.iloc[-TEST_PERIODS_SHAMPOO:]['yhat1']
metrics_neural = get_metrics(y_true, y_pred)
results.append({"Dataset": dataset_name, "Model": "NeuralProphet", **metrics_neural})

# --- Model 4: Auto-ARIMA ---
# Basit, mevsimsel olmayan ARIMA (m=1)
m_arima = auto_arima(train_df['y'], seasonal=False, m=1, suppress_warnings=True, stepwise=True)
y_pred = m_arima.predict(n_periods=TEST_PERIODS_SHAMPOO)
metrics_arima = get_metrics(y_true, y_pred)
results.append({"Dataset": dataset_name, "Model": "Auto-ARIMA", **metrics_arima})

print("Shampoo Sales tamamlandı.")
print("\nTüm karşılaştırmalar bitti.")

Veri Seti 1: Air Passengers çalıştırılıyor...


22:00:44 - cmdstanpy - INFO - Chain [1] start processing
22:00:44 - cmdstanpy - INFO - Chain [1] done processing
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
WARNING - (py.warnings._showwarnmsg) - /usr/local/lib/python3.11/dist-packages/neuralprophet/df_utils.py:1152: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  converted_ds = pd.to_datetime(ds_col, utc=True).view(dtype=np.int64)

INFO - (NP.df_utils._infer_frequency) - Major frequency MS corresponds to [99.167]% of the data.
WARNING - (py.warnings._showwarnmsg) - /usr/local/lib/python3.11/dist-packages/neuralprophet/df_utils.py:1152: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  converted_ds = pd.to_datetime(ds_col, utc=True).view(dtype=np.int64)

WARNING - (py.warnings._showwa

Training: |          | 0/? [00:00<?, ?it/s]

WARNING - (NP.config.set_lr_finder_args) - Learning rate finder: The number of batches (8) is too small than the required number                     for the learning rate finder (208). The results might not be optimal.
2025-10-22 22:00:47.923750: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761170448.231807      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761170448.316375      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Finding best initial lr:   0%|          | 0/208 [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

WARNING - (py.warnings._showwarnmsg) - /usr/local/lib/python3.11/dist-packages/neuralprophet/df_utils.py:1152: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  converted_ds = pd.to_datetime(ds_col, utc=True).view(dtype=np.int64)

INFO - (NP.df_utils._infer_frequency) - Major frequency MS corresponds to [96.667]% of the data.
WARNING - (py.warnings._showwarnmsg) - /usr/local/lib/python3.11/dist-packages/neuralprophet/df_utils.py:1152: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  converted_ds = pd.to_datetime(ds_col, utc=True).view(dtype=np.int64)

WARNING - (py.warnings._showwarnmsg) - /usr/local/lib/python3.11/dist-packages/neuralprophet/df_utils.py:1152: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  converted_ds = p

Predicting: |          | 0/? [00:00<?, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


Shampoo Sales tamamlandı.

Tüm karşılaştırmalar bitti.


In [6]:
# Sonuçları DataFrame'e dönüştür
results_df = pd.DataFrame(results)

# MAPE'yi yüzde olarak göstermek için formatla
results_df['MAPE'] = results_df['MAPE'].apply(lambda x: f"{x:.2%}")

# Daha iyi okunabilirlik için RMSE ve MAE'yi yuvarla
results_df['RMSE'] = results_df['RMSE'].round(2)
results_df['MAE'] = results_df['MAE'].round(2)

# Tabloyu göster
print("\n--- Model Karşılaştırma Sonuçları ---")
print(results_df.to_markdown(index=False))


--- Model Karşılaştırma Sonuçları ---
| Dataset        | Model         |   RMSE |    MAE | MAPE   |
|:---------------|:--------------|-------:|-------:|:-------|
| air_passengers | OptiProphet   |  27.22 |  23.55 | 5.30%  |
| air_passengers | Prophet       |  76.47 |  56.62 | 11.50% |
| air_passengers | NeuralProphet |  26.98 |  22.63 | 4.75%  |
| air_passengers | Auto-ARIMA    |  14.11 |  10.62 | 2.36%  |
| shampoo_sales  | OptiProphet   | 147.28 | 140.71 | 24.83% |
| shampoo_sales  | Prophet       | 643.41 | 560.53 | 96.30% |
| shampoo_sales  | NeuralProphet | 158.13 | 147.6  | 26.30% |
| shampoo_sales  | Auto-ARIMA    | 142.05 | 120.18 | 19.76% |
